In [ ]:
from typing import Any, Optional
from arcengine import GameAction


from arcengine import FrameDataRaw

from arc import MyArcSession
from arc_agi import OperationMode
from arc_agi import Arcade
import logging
import numpy as np
from dotenv import load_dotenv
from PIL import Image
from sandbox import SandboxOrchestrator
from google.genai import types
from google import genai
from tools import TOOL_LIST, SYSTEM_PROMPT, TAKE_ACTION
from agent import JackAgent
import io

load_dotenv()


PALETTE = np.array(
    [
        [0xFF, 0xFF, 0xFF],
        [0xCC, 0xCC, 0xCC],
        [0x99, 0x99, 0x99],
        [0x66, 0x66, 0x66],
        [0x33, 0x33, 0x33],
        [0x00, 0x00, 0x00],
        [0xE5, 0x3A, 0xA3],
        [0xFF, 0x7B, 0xCC],
        [0xF9, 0x3C, 0x31],
        [0x1E, 0x93, 0xFF],
        [0x88, 0xD8, 0xF1],
        [0xFF, 0xDC, 0x00],
        [0xFF, 0x85, 0x1B],
        [0x92, 0x12, 0x31],
        [0x4F, 0xCC, 0x30],
        [0xA3, 0x56, 0xD6],
    ],
    dtype=np.uint8,
)


def render_frame(frame: np.ndarray) -> Image.Image:
    rgb = PALETTE[np.clip(np.asarray(frame, dtype=np.uint8), 0, 15)]
    return Image.fromarray(rgb).resize((512, 512), Image.NEAREST)


logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S"
)


def current_state_prompt(obs: FrameDataRaw) -> str:
    return f"""Currently playing game: {obs.game_id}. On level #{obs.levels_completed} out of #{obs.win_levels}.

"""


def starting_state_prompt(obs: FrameDataRaw) -> str:
    res = f"""Starting the following game: {obs.game_id}

"""

    return (
        f"New game started. Initial state:\n{json.dumps(obs, indent=2)}\n\n"
        f"The full grid is in /home/agent/state.json. "
        f"Start by calling render_board to see the board, then take actions to explore."
    )

In [12]:

arcade = Arcade(operation_mode=OperationMode("normal"))
scorecard_id: str = arcade.open_scorecard(tags=["jackagent"])
arc_session = MyArcSession(
    game_id=["ls20", "ft09", "vc33"][-1], arcade=arcade, scorecard_id=scorecard_id
)
sbx = SandboxOrchestrator()
print(sbx.bash("echo sandbox ready"))

agent = JackAgent(sbx=sbx, arc_session=arc_session)


agent.contents = []

2026-04-13 04:17:43 | INFO | Successfully fetched 25 environment(s) from API


04:17:43 Successfully fetched 25 environment(s) from API


2026-04-13 04:17:43 | INFO | Created new scorecard: 1993e38f-7cc3-450b-a0e8-687a22dbbaf4


04:17:43 Created new scorecard: 1993e38f-7cc3-450b-a0e8-687a22dbbaf4


2026-04-13 04:17:43 | INFO | Successfully fetched metadata for game vc33


...

2026-04-13 04:17:43 | INFO | Recording to recordings/1993e38f-7cc3-450b-a0e8-687a22dbbaf4/vc33-9851e02b-4044a980-b47a-403d-b141-466d9ffb2bf4.jsonl


04:17:43 Recording to recordings/1993e38f-7cc3-450b-a0e8-687a22dbbaf4/vc33-9851e02b-4044a980-b47a-403d-b141-466d9ffb2bf4.jsonl


2026-04-13 04:17:43 | INFO | Successfully loaded game class Vc33 from environment_files/vc33/9851e02b/vc33.py


04:17:43 Successfully loaded game class Vc33 from environment_files/vc33/9851e02b/vc33.py


sandbox ready


In [13]:
agent.contents = [
    types.Content(role="user", parts=[types.Part(text=(
        f"Game: {arc_session.obs.game_id} | "
        f"Score: {arc_session.obs.levels_completed}/{arc_session.obs.win_levels} | "
        f"Actions: {[GameAction.from_id(a).name for a in arc_session.obs.available_actions]}\n\n"
        f"Current grid is in /home/agent/state.json. "
        f"Full action history is in /home/agent/replay.jsonl.\n\n"
        f"Workflow:\n"
        f"1. Render: python3 -c 'from helpers import *; render_board()' then view_image('board.png')\n"
        f"2. Analyze the grid — find objects, colors, structure, symmetries\n"
        f"3. Take an action, then render + view again to see what changed\n"
        f"4. Write your theory of the rules to notes.md — update it as you learn\n"
        f"5. Use matplotlib to annotate images — draw boxes, arrows, highlights to verify your understanding\n"
        f"6. Never guess randomly. Build a theory, test it, refine it.\n\n"
        f"You can create any visualization you want with matplotlib/pillow and see it with view_image. "
        f"Solve this game."
    ))])
]

agent.contents.append(types.Content(role="user", parts=[
    types.Part(text="solve quickly")]))

for _ in range(100):
    calls = agent.generate_response()
    if not calls:
        last = agent.contents[-1]
        for p in last.parts:
            if hasattr(p, "text") and p.text:
                print(f"[thinking] {p.text[:300]}\n")
    for c in calls:
        print(f"[{c['name']}] {c['args']}")
        print(f"  → {str(c['result'])[:300]}\n")
        if c["name"] == "take_action" and agent.arc_session.obs.frame:
            display(render_frame(agent.arc_session.obs.frame[-1]))
        if c["name"] == "view_image":
            try:
                raw = agent.sbx.read_file(
                    agent.sbx._resolve(c["args"]["file_path"]))
                display(Image.open(io.BytesIO(raw)))
            except:
                pass

04:17:51 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[bash] {'command': "python3 -c 'from helpers import *; render_board()'"}
  → no output



04:17:53 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[view_image] {'file_path': 'board.png'}
  → Displaying board.png



04:17:56 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


...

04:18:00 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[bash] {'command': 'cat << \'EOF\' > analyze.py\nfrom helpers import *\nimport numpy as np\n\ngrid = load_grid()\n\nfor c in np.unique(grid):\n    coords = find_color(grid, c)\n    print(f"Color {c}: {len(coords)} pixels")\n    if len(coords) < 100:\n        print(f"  {coords[:5]} ... {coords[-5:]}")\n\nEOF\npython3 analyze.py'}
  → Color 0: 1216 pixels
Color 3: 2580 pixels
Color 4: 16 pixels
  [(44, 48), (44, 49), (45, 48), (45, 49), (46, 46)] ... [(47, 49), (48, 48), (48, 49), (49, 48), (49, 49)]
Color 5: 168 pixels
Color 7: 64 pixels
  [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)] ... [(0, 59), (0, 60), (0, 61), (0, 62), (0, 63)]



04:18:07 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[bash] {'command': 'cat << \'EOF\' > get_shapes.py\nfrom helpers import *\nimport numpy as np\n\ngrid = load_grid()\n\ndef get_bboxes(c):\n    coords = np.argwhere(grid == c)\n    if len(coords) == 0: return None\n    min_r, min_c = coords.min(axis=0)\n    max_r, max_c = coords.max(axis=0)\n    return (min_r, max_r, min_c, max_c)\n\nfor c in np.unique(grid):\n    if c in [0, 3]: continue\n    bbox = get_bboxes(c)\n    print(f"Color {c}: bounding box (r_min={bbox[0]}, r_max={bbox[1]}, c_min={bbox[2]}, c_max={bbox[3]})")\nEOF\npython3 get_shapes.py'}
  → Color 4: bounding box (r_min=44, r_max=49, c_min=46, c_max=49)
Color 5: bounding box (r_min=28, r_max=31, c_min=20, c_max=63)
Color 7: bounding box (r_min=0, r_max=0, c_min=0, c_max=63)
Color 9: bounding box (r_min=24, r_max=35, c_min=60, c_max=63)
Color 11: bounding box (r_min=28, r_max=49, c_min=3



KeyboardInterrupt: 